# Semaine 9 — Jour 1 : Chunking, provenance et cycle documentaire

Découper un document en passages bornés avec chevauchement contrôlé.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J1_chunking.ipynb` et `notebooks/week09/teacher/S9_J1_chunking_teacher.ipynb`.

Critère de réussite : Relier une version documentaire à ses chunks sans perdre la provenance.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Découper un document en passages bornés avec chevauchement contrôlé.
- Conserver des offsets vérifiables dans la source et des identifiants stables.
- Relier une version documentaire à ses chunks sans perdre la provenance.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J1 — Chunking, provenance et cycle documentaire


## 1. Le passage devient l’unité de recherche
Un système de connaissances ne retrouve pas seulement des documents : il prépare des passages assez précis pour répondre et assez complets pour garder leur sens. Découper trop finement isole une phrase de sa condition; découper trop largement dilue le signal et consomme du contexte. Le bon découpage dépend de la structure du corpus et des questions attendues.

Le fil rouge utilise huit documents fictifs de backend et DevOps. Sept appartiennent à team-a; un document voisin appartient à team-b. Les procédures et valeurs sont des conventions du lab. L’objectif est de construire la chaîne documentaire et ses contrôles, sans faire passer ces notes pour une documentation d’un système bancaire réel.

## 2. Définir un contrat documentaire
Chaque document porte doc_id, tenant, version, title, source et text. Le tenant représente un périmètre déjà autorisé par le serveur dans une architecture réelle. Il ne doit pas venir d’un choix libre d’un client qui obtiendrait ainsi les droits d’un voisin. La version est un entier positif; une modification du contenu exige une nouvelle version.

Le document constitue la source de vérité. Les chunks et vecteurs sont des projections reconstruites. Une suppression de source doit aussi retirer ses passages de l’index. Un index oublié après une suppression peut continuer à exposer une ancienne information même si le fichier d’origine a disparu.

## 3. Algorithme local
chunk_document identifie les paragraphes puis crée des fenêtres de mots séparés par des espaces. Une fenêtre longue de max_words avance de max_words−overlap jusqu’à la fin du paragraphe. La garde overlap < max_words garantit la progression. Un paragraphe court produit un seul chunk et ne doit pas créer de doublon final.

Les mots de découpage ne sont pas les tokens d’un modèle. Les offsets start/end sont des indices de caractères Python dans le texte Unicode original. La propriété à vérifier est source[start:end] == chunk.text. La normalisation utilisée plus tard pour rechercher n’altère pas le texte cité ni ses positions.

## 4. Taille, chevauchement et structure
Le chevauchement protège un peu les relations autour d’une frontière, mais augmente le nombre de passages et la redondance. Il ne garantit pas qu’une condition éloignée reste présente. Mesurer les questions qui échouent avant de choisir une valeur universelle. Des listes, tableaux ou blocs de code peuvent demander un découpage différent de la prose.

Le chunker fourni garde aussi les titres isolés comme petits chunks. Cette simplicité révèle un bruit utile à observer : un titre peut être très bien classé sans fournir la réponse. Le mode extractif final ignore les chunks constitués d’un paragraphe de titre. Une amélioration serait de rattacher le titre au paragraphe tout en gardant des offsets exacts et un budget explicite.

## 5. Identité et évolution
L’identifiant d’un chunk dérive de tenant, doc_id, version, offsets et texte. Refaire le même découpage produit les mêmes identifiants. Modifier le chevauchement ou la version peut produire un autre ensemble. Le titre et la source restent dans les métadonnées; une modification du document avec la même version est refusée par le store.

L’ingestion prépare les vecteurs avant de remplacer l’ensemble des chunks dans une transaction. Si l’encodage ou l’écriture échoue, les passages précédents restent disponibles. Ce remplacement atomique concerne un document dans le processus SQLite; il ne prétend pas synchroniser un stockage documentaire externe et un cluster vectoriel.

## 6. Preuve du jour
Vérifier les offsets sur un document accentué, un paragraphe long et un texte sans ligne vide. Compter le surplus créé par overlap. Conserver un exemple où une question exige deux passages afin de montrer la limite d’un découpage purement mécanique. Le chunking ne résout ni pertinence ni autorisation : il prépare leur unité de travail.

## À retenir
Un chunk utile a une taille maîtrisée, une provenance vérifiable et un cycle de vie relié à la source. Une chaîne de recherche ne peut pas réparer une condition perdue lors du découpage.


## Diagramme du jour

```mermaid
flowchart TD
 A["Source et version"] --> B["Paragraphes"]
 B --> C["Fenêtres bornées"]
 C --> D["Offsets et identifiants"]
 D --> E["Projections d’index"]
 A --> F["Contrôle source[start:end]"]
 D --> F
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week09.demo import documents
from examples.week09.knowledge import chunk_document, ContractError
doc = documents()[0]
chunks = chunk_document(doc, max_words=24, overlap=4)
assert chunks and all(doc["text"][c["start"]:c["end"]] == c["text"] for c in chunks)
assert len({c["chunk_id"] for c in chunks}) == len(chunks)
print({"chunks": len(chunks), "first": chunks[0]})

# Lab J1 — Chunking, provenance et cycle documentaire

Depuis la racine du dépôt :

```bash
python3 book/week09/day01/labs/run.py
python3 -m examples.week09.demo --day 1
```

Résultat attendu : **plusieurs chunks, offsets_valid=true et premier passage inspectable**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Calculer les fenêtres

Pour dix mots, max_words=4 et overlap=1, donner les intervalles de mots produits.

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Expliquer les offsets

Pourquoi normaliser les accents avant de calculer les offsets rendrait-il certaines citations incorrectes ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Faire évoluer une source

Quelles projections faut-il remplacer quand un document change de version ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Comment choisir la taille des chunks ?

## Question 2

Pourquoi conserver la version dans une citation ?


# Challenge

Créer un document de dix mots avec une seule ligne. Vérifier les trois fenêtres, leurs offsets et la stabilité des identifiants après un second découpage identique.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Références — J1

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
